# Feature Engineering

This notebook focuses on creating business-driven analytical features and operational KPIs to support supply chain monitoring, logistics analysis and executive dashboard reporting.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv(
    "../data/processed/cleaned_supply_chain.csv"
)

In [3]:
df["order_date_dateorders"] = pd.to_datetime(
    df["order_date_dateorders"]
)

df["shipping_date_dateorders"] = pd.to_datetime(
    df["shipping_date_dateorders"]
)

# Feature Engineering

Creating derived business metrics and operational indicators to support:
- shipment performance monitoring
- logistics risk analysis
- profitability analysis
- executive KPI reporting
- control tower dashboards

## Shipment Delay Feature

Calculating the difference between actual shipping duration and scheduled shipping duration to measure shipment delays and operational efficiency.

In [4]:
df["shipping_delay_days"] = (
    df["days_for_shipping_real"]
    -
    df["days_for_shipment_scheduled"]
)

In [5]:
df[
    [
        "days_for_shipping_real",
        "days_for_shipment_scheduled",
        "shipping_delay_days"
    ]
].head()

,days_for_shipping_real,days_for_shipment_scheduled,shipping_delay_days
0,3,4,-1
1,5,4,1
2,4,4,0
3,3,4,-1
4,2,4,-2


## Delivery Performance Classification

Creating a shipment performance category based on shipping delay duration to support operational monitoring and executive reporting.

In [6]:
df["delivery_performance"] = np.where(
    df["shipping_delay_days"] > 0,
    "Delayed",
    np.where(
        df["shipping_delay_days"] == 0,
        "On Time",
        "Early Delivery"
    )
)

In [7]:
df[
    [
        "shipping_delay_days",
        "delivery_performance"
    ]
].head(10)

,shipping_delay_days,delivery_performance
0,-1,Early Delivery
1,1,Delayed
2,0,On Time
3,-1,Early Delivery
4,-2,Early Delivery
5,2,Delayed
6,1,Delayed
7,1,Delayed
8,1,Delayed
9,1,Delayed


## Time Intelligence Features

Creating date hierarchy features from the order date to support trend analysis, seasonality detection and executive KPI reporting.

In [8]:
df["order_year"] = df["order_date_dateorders"].dt.year

df["order_month"] = df["order_date_dateorders"].dt.month

df["order_month_name"] = df["order_date_dateorders"].dt.month_name()

df["order_quarter"] = df["order_date_dateorders"].dt.quarter

df["order_day_of_week"] = df["order_date_dateorders"].dt.day_name()

In [9]:
df[
    [
        "order_date_dateorders",
        "order_year",
        "order_month",
        "order_month_name",
        "order_quarter",
        "order_day_of_week"
    ]
].head()

,order_date_dateorders,order_year,order_month,order_month_name,order_quarter,order_day_of_week
0,2018-01-31 22:56:00,2018,1,January,1,Wednesday
1,2018-01-13 12:27:00,2018,1,January,1,Saturday
2,2018-01-13 12:06:00,2018,1,January,1,Saturday
3,2018-01-13 11:45:00,2018,1,January,1,Saturday
4,2018-01-13 11:24:00,2018,1,January,1,Saturday


## Logistics KPI Flags

Creating binary indicators for shipment performance monitoring and KPI calculations.

In [10]:
df["delayed_flag"] = np.where(
    df["shipping_delay_days"] > 0,
    1,
    0
)

df["on_time_flag"] = np.where(
    df["shipping_delay_days"] <= 0,
    1,
    0
)

In [11]:
df[
    [
        "shipping_delay_days",
        "delayed_flag",
        "on_time_flag"
    ]
].head()

,shipping_delay_days,delayed_flag,on_time_flag
0,-1,0,1
1,1,1,0
2,0,0,1
3,-1,0,1
4,-2,0,1


## Profitability Features

Creating profitability indicators to identify profitable, low-margin and loss-making orders. These features will support margin analysis, discount effectiveness studies and executive profitability reporting.

In [12]:
df["loss_order_flag"] = np.where(
    df["order_profit_per_order"] < 0,
    1,
    0
)

In [13]:
df["order_item_discount_rate"].describe()

count    180519.000000
mean          0.101668
std           0.070415
min           0.000000
25%           0.040000
50%           0.100000
75%           0.160000
max           0.250000
Name: order_item_discount_rate, dtype: float64

In [14]:
df["order_profit_per_order"].describe()

count    180519.000000
mean         21.974989
std         104.433526
min       -4274.979980
25%           7.000000
50%          31.520000
75%          64.800003
max         911.799988
Name: order_profit_per_order, dtype: float64

## Discount Analysis Features

Creating discount-related indicators to identify orders receiving unusually high discounts that may impact profitability.

In [15]:
df["high_discount_flag"] = np.where(
    df["order_item_discount_rate"] >= 0.15,
    1,
    0
)

## Profitability Segmentation

Classifying orders into profitability bands using profit distribution quartiles to support profitability analysis and executive reporting.

In [16]:
df["profit_category"] = np.where(
    df["order_profit_per_order"] < 0,
    "Loss",
    np.where(
        df["order_profit_per_order"] <= 7,
        "Low Profit",
        np.where(
            df["order_profit_per_order"] <= 64.8,
            "Medium Profit",
            "High Profit"
        )
    )
)

In [17]:
df[
    [
        "order_profit_per_order",
        "profit_category"
    ]
].head(10)

,order_profit_per_order,profit_category
0,91.250000,High Profit
1,-249.089996,Loss
2,-247.779999,Loss
3,22.860001,Medium Profit
4,134.210007,High Profit
5,18.580000,Medium Profit
6,95.180000,High Profit
7,68.430000,High Profit
8,133.720001,High Profit
9,132.149994,High Profit


In [18]:
df["profit_category"].value_counts()

profit_category
Medium Profit    90218
High Profit      45167
Loss             33784
Low Profit       11350
Name: count, dtype: int64

## Profitability Segmentation Findings

The profit distribution was segmented using data-driven thresholds derived from quartile analysis.

Key observations:

- Approximately 50% of orders fall into the Medium Profit category.
- Nearly 19% of orders are loss-making and may require operational investigation.
- High Profit orders represent roughly the top 25% of transactions.
- The distribution is well balanced and suitable for profitability analysis and dashboard reporting.

In [19]:
df["shipping_delay_days"].describe()

count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: shipping_delay_days, dtype: float64

In [20]:
df["shipping_delay_days"].value_counts().sort_index()

shipping_delay_days
-2    21666
-1    21700
 0    33753
 1    60647
 2    28718
 3     7052
 4     6983
Name: count, dtype: int64

## Severe Delay Indicator

Creating a flag to identify shipments delayed by two or more days. These shipments represent elevated operational risk and may indicate logistics bottlenecks.

In [21]:
df["severe_delay_flag"] = np.where(
    df["shipping_delay_days"] >= 2,
    1,
    0
)

## Shipment Risk Classification

Classifying shipments into operational risk categories based on delivery delay severity.

In [22]:
df["shipment_risk_level"] = np.select(
    [
        df["shipping_delay_days"] <= 0,
        df["shipping_delay_days"] == 1,
        df["shipping_delay_days"] >= 2
    ],
    [
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ],
    default="Unknown"
)

In [23]:
df["shipment_risk_level"].value_counts()

shipment_risk_level
Low Risk       77119
Medium Risk    60647
High Risk      42753
Name: count, dtype: int64

## Shipment Risk Analysis Findings

Shipment risk was classified using shipping delay severity.

Key observations:

- Approximately 43% of shipments are classified as Low Risk.
- Around 34% of shipments fall into the Medium Risk category.
- Nearly 24% of shipments are categorized as High Risk and may require operational investigation.

These risk classifications will support logistics monitoring, bottleneck identification and executive KPI reporting.

In [24]:
df["sales"].describe()

count    180519.000000
mean        203.772096
std         132.273077
min           9.990000
25%         119.980003
50%         199.919998
75%         299.950012
max        1999.989990
Name: sales, dtype: float64

## Revenue Intelligence Features

Creating revenue-based classifications to identify high-value transactions and support executive revenue analysis.

In [25]:
df["high_value_order_flag"] = np.where(
    df["sales"] >= 300,
    1,
    0
)

In [26]:
df["sales_band"] = np.select(
    [
        df["sales"] < 120,
        (df["sales"] >= 120) & (df["sales"] < 300),
        df["sales"] >= 300
    ],
    [
        "Low Value",
        "Medium Value",
        "High Value"
    ],
    default="Unknown"
)

In [27]:
df["sales_band"].value_counts()

sales_band
Medium Value    107382
Low Value        47181
High Value       25956
Name: count, dtype: int64

In [28]:
df["high_value_order_flag"].value_counts()

high_value_order_flag
0    154563
1     25956
Name: count, dtype: int64

## Revenue Intelligence Findings

Revenue segmentation was created using sales distribution thresholds.

Key observations:

- Approximately 60% of orders fall within the Medium Value category.
- Around 26% of orders are classified as Low Value.
- Roughly 14% of orders are categorized as High Value.
- High Value Orders represent an important segment for revenue-focused operational monitoring and customer service prioritization.

These features will support revenue analysis, risk assessment and executive KPI reporting.

# Feature Engineering Summary

The following business-driven features were engineered to support supply chain monitoring, profitability analysis, operational risk assessment and executive reporting.

## Time Intelligence Features
- order_year
- order_month
- order_month_name
- order_quarter
- order_day_of_week

## Logistics Features
- shipping_delay_days
- delivery_performance
- delayed_flag
- on_time_flag
- severe_delay_flag

## Risk Features
- shipment_risk_level

## Profitability Features
- loss_order_flag
- high_discount_flag
- profit_category

## Revenue Features
- high_value_order_flag
- sales_band

In [29]:
df.to_csv(
    "../data/processed/featured_supply_chain.csv",
    index=False
)

# Feature Engineering Completion Summary

A business-focused feature layer was created to support supply chain analytics, logistics monitoring, profitability analysis, risk assessment and executive reporting.

The engineered features include:

### Time Intelligence
- order_year
- order_month
- order_month_name
- order_quarter
- order_day_of_week

### Logistics & Delivery
- shipping_delay_days
- delivery_performance
- delayed_flag
- on_time_flag
- severe_delay_flag

### Risk Monitoring
- shipment_risk_level

### Profitability
- loss_order_flag
- high_discount_flag
- profit_category

### Revenue Intelligence
- high_value_order_flag
- sales_band

The resulting dataset is now ready for SQL warehouse modeling and dashboard development.

# MySQL Compatible Export

The MySQL Workbench Import Wizard has issues with non-ASCII characters and large files.

To improve compatibility, all text fields are converted to ASCII characters before export.

Examples:

- México → Mexico
- Bogotá → Bogota
- São Paulo → Sao Paulo

This export is intended specifically for SQL warehouse ingestion.

In [32]:
for col in df.select_dtypes(include="object").columns:
    non_ascii = df[col].astype(str).str.contains(
        r"[^\x00-\x7F]",
        regex=True
    ).sum()

    if non_ascii > 0:
        print(col, ":", non_ascii)

/var/folders/f_/hf2_lrtj217g14m6qshc00180000gn/T/ipykernel_12923/1900395947.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


order_city : 15208
order_country : 37430
order_state : 35310


# Convert Text Columns to ASCII

Special characters are removed from all text columns to maximize compatibility with SQL import tools.

In [33]:
import unicodedata

for col in df.select_dtypes(include="object").columns:

    df[col] = (
        df[col]
        .astype(str)
        .apply(
            lambda x:
            unicodedata
            .normalize("NFKD", x)
            .encode("ascii", "ignore")
            .decode("ascii")
        )
    )

print("ASCII conversion completed.")

/var/folders/f_/hf2_lrtj217g14m6qshc00180000gn/T/ipykernel_12923/1676845350.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


ASCII conversion completed.


# Validate ASCII Conversion

Verify that all non-ASCII characters have been removed successfully.

In [34]:
for col in df.select_dtypes(include="object").columns:
    non_ascii = df[col].astype(str).str.contains(
        r"[^\x00-\x7F]",
        regex=True
    ).sum()

    if non_ascii > 0:
        print(col, ":", non_ascii)

/var/folders/f_/hf2_lrtj217g14m6qshc00180000gn/T/ipykernel_12923/1900395947.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


# Export SQL Compatible Dataset

Create final dataset for MySQL warehouse loading.

In [35]:
df.to_csv(
    "../data/processed/featured_supply_chain_ascii.csv",
    index=False,
    encoding="ascii"
)

print("ASCII export completed successfully.")

ASCII export completed successfully.
